# 1. Synchronous Programming

Definition: Code runs line by line; each operation must finish before the next starts.

### Behavior in web APIs:

- If one request takes time (e.g., database query, API call), other requests wait.
- Can cause performance bottlenecks under heavy load.

   **Example (sync):**
   ```py
    def get_data():
       time.sleep(5)  # Simulates a slow DB/API call
    return {"message": "Done"}

- During the 5 seconds, no other request can process on this worker thread.

---

# 2. Why Async Matters in FastAPI

FastAPI supports async route handlers to avoid blocking.

#### Benefits:

- Handles thousands of concurrent requests efficiently.
- Makes I/O-heavy operations (like DB queries, external API calls) non-blocking.
- Improves throughput without adding threads.

> Scenario: You fetch data from a slow API or database. Using sync code → your worker thread waits. Using async → the event loop can handle other requests in the meantime.

---

# 3. async def vs def in Route Handlers
| Type       | Description                     | When to Use                                                      |
|------------|---------------------------------|------------------------------------------------------------------|
| `def`      | Standard synchronous function   | Use for CPU-bound operations or very fast calls that don’t block I/O. |
| `async def`| Asynchronous coroutine          | Use for I/O-bound operations like database queries, network requests, file reads, etc. |

#### Rule of Thumb:

- If your route mainly waits for I/O, use async def.
- If it’s CPU-heavy or calls only sync code, def is fine.

---

# 4. How FastAPI Runs Async & Sync Code

- **Async (async def):** Uses Python’s asyncio event loop. FastAPI does not block; other requests can run while awaiting.
- **Sync (def):** FastAPI wraps it in a threadpool so the main event loop isn’t blocked, but each thread is still synchronous.

> FastAPI transparently mixes async and sync routes, but using async consistently for I/O-heavy tasks is more efficient.

---

# 5. Non-blocking I/O (Basic Idea)

- **Problem:** Waiting for external resources (DB/API) blocks the thread.
- **Solution:** Async I/O lets you "pause" the operation and let the event loop do other work.
- **Analogy:** You order food at a restaurant (await your meal). Instead of standing idle, you help other customers while waiting.

   **Example with async DB call:**
   ```py

     from databases import Database
     db = Database("sqlite:///test.db")
     async def get_users():
         query = "SELECT * FROM users"
         return await db.fetch_all(query)  # Non-blocking
   
- While db.fetch_all waits, other requests are processed.

---

# ✅ Key Takeaways

- Use async for I/O-bound work; use def for CPU-bound work.
- Async allows non-blocking concurrency, improving API responsiveness.
- FastAPI manages both sync and async routes efficiently.
- Database calls, HTTP requests, or file operations are common async candidates.